In [1]:
# conda activate genomic_tools

import os
import json
import pickle
import pandas as pd
from collections import defaultdict

## Load interproscan results

In [2]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [3]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [4]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

## Map events to interproscan results

(Get the transcript associated with each significant event)

In [ ]:
# get all significant splicing events

signif_events = []
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        for idx, _ in signif_exons_df.iterrows():
            if idx not in signif_events:
                signif_events.append(idx)

In [6]:
# map events to interproscan results

with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

In [8]:
columns = ['protein_accession', 'sequence_length', 'analysis', 
           'signature_description', 'start', 'stop', 'interpro_description']
analyses_to_exclude = ['NCBIFAM', 'SFLD'] # these tools are for full-length protein classification so it doesn't make sense to consider them as overlapping with a single exon
ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

# index ipr by protein_accession once
ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

def overlaps_skip(df, aa_start):
    return df[(df['start'] <= aa_start) & (df['stop'] >= aa_start)]

event_interproscan_map = defaultdict(dict)
  
for ev, rec in event_protein_map.items():
    # inclusion: direct lookup instead of boolean mask
    rec_incl = rec['inclusion']
    incl_df = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    overlap_df = incl_df[(incl_df['start'] <= rec_incl['aa_end']) & 
                         (incl_df['stop']  >= rec_incl['aa_start'])]
    if overlap_df.empty:
        continue
    event_interproscan_map[ev]['inclusion'] = overlap_df.assign(
        aa_start = rec_incl['aa_start'],
        aa_end = rec_incl['aa_end'],
        exon_cds_start = rec_incl['exon_cds_start'],
        exon_cds_end = rec_incl['exon_cds_end'],
        frame_preserving = rec_incl['frame_preserving'],
        clean_start = rec_incl['clean_start'],
        clean_end = rec_incl['clean_end'],
    ).reset_index(drop=True)

    # real skip
    if rec.get('real_skip'):
        skip_df = ipr_grouped.get(rec['real_skip'])
        if skip_df is not None:
            junction_aa = rec.get('real_skip_junction_aa', rec_incl['aa_start'])
            s = overlaps_skip(skip_df, junction_aa).assign(
                truncation_aa = junction_aa
            )
            if not s.empty:
                event_interproscan_map[ev]['real_skip'] = s.reset_index(drop=True)

    # synthetic skip
    if rec.get('synthetic_skip'):
        synth_df = ipr_grouped.get(rec['synthetic_skip'])
        if synth_df is not None:
            s = overlaps_skip(synth_df, rec_incl['aa_start'])
            if not s.empty:
                event_interproscan_map[ev]['synthetic_skip'] = s.reset_index(drop=True)

    # junction siblings
    if rec.get('exon_diff_junction_siblings'):
        frames = []
        for sib in rec['exon_diff_junction_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_junction_siblings'] = pd.concat(frames, ignore_index=True)

    # boundary siblings
    if rec.get('exon_diff_boundary_siblings'):
        frames = []
        for sib in rec['exon_diff_boundary_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_boundary_siblings'] = pd.concat(frames, ignore_index=True)

In [11]:
with open('data/event_interproscan_map.pkl', "wb") as file:
    pickle.dump(event_interproscan_map, file)

## Merge interproscan results with significant splicing event info.

In [ ]:
# merge cell type-specific events with InterProScan results

signif_event_interproscan_map = dict() 
signif_event_interproscan_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {
            ev: event_interproscan_map[ev] for ev in signif_events_df.index 
            if ev in event_interproscan_map
        }

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events with InterProScan results
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
        signif_event_interproscan_map[ctype] = df
        
        # summarize interpro results for significant splicing events
        signif_event_interproscan_summary[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            chr=('chr', lambda x: ' | '.join(x.unique())),
            exon_start=('exon_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_end=('exon_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_len=('exon_len', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_start=('exon_cds_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_end=('exon_cds_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_start=('aa_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_end=('aa_end', lambda x: ' | '.join(map(str, x.unique()))),
            domain_start=('start', lambda x: ' | '.join(map(str, x.unique()))),
            domain_stop=('stop', lambda x: ' | '.join(map(str, x.unique()))),
            protein_sequence_length=('sequence_length', lambda x: ' | '.join(map(str, x.unique()))),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(x.unique())),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo
VLMC
Endo


In [ ]:
with open("data/signif_event_interproscan_map.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_map, file)
    
with open("data/signif_event_interproscan_summary.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_summary, file)

Preview

In [ ]:
signif_event_interproscan_summary['All_GABAergic'][signif_event_interproscan_summary['All_GABAergic']['is_specific'] == "True"].sort_values('n_analyses', ascending=False).head(10)

,event_id,bucket,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,...,exon_aa_start,exon_aa_end,domain_start,domain_stop,protein_sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
2023,ENSG00000092096_ProteinCoding_2,inclusion,0.2085403902319068,True,SLC22A17,ENST00000354772,chr14,23351752,23351855,104,...,200.0,234.0,177 | 150 | 174 | 217 | 207 | 230 | 1 | 215 | ...,599 | 587 | 597 | 229 | 240 | 206 | 596 | 245 ...,631,False,10,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,MFS general substrate transporter like domains...,MFS transporter superfamily | - | Major facili...
2185,ENSG00000099204_ProteinCoding_2,exon_diff_boundary_siblings,-0.3991881867582086,True,ABLIM1,ENST00000533213,chr10,114447880,114448026,147,...,nan,nan,216 | 284 | 88 | 152 | 702 | 86 | 700 | 217 | ...,283 | 349 | 151 | 215 | 778 | 150 | 216 | 350 ...,778,nan,10,CATH-Gene3D | CATH-FunFam | CDD | MobiDB-lite ...,Cysteine Rich Protein | Villin headpiece domai...,- | Villin headpiece domain superfamily | Puta...
595,ENSG00000049323_ProteinCoding_2,inclusion,-0.2017625375727214,True,LTBP1,ENST00000404816,chr2,33342838,33342963,126,...,1243.0,1285.0,1203 | 1244 | 1202 | 872 | 1201 | 1285 | 1224 ...,1243 | 1286 | 1328 | 1276 | 1314 | 1284 | 1316...,1721,True,10,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Laminin | latent-transforming growth factor be...,- | Complement Clr-like EGF domain | EGF-like ...
10286,ENSG00000179520_ProteinCoding_1,inclusion,0.3529908454564064,True,SLC17A8,ENST00000323346,chr12,100402596,100402745,150,...,301.0,350.0,310 | 79 | 117 | 271 | 334 | 345 | 73 | 75 | 315,508 | 502 | 463 | 309 | 344 | 333 | 370 | 503 ...,589,True,9,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,MFS general substrate transporter like domains...,MFS transporter superfamily | - | Major facili...
1960,ENSG00000090621_ProteinCoding_3,exon_diff_boundary_siblings,0.1962318802688724,True,PABPC4,ENST00000678625,chr1,39564686,39564773,88,...,nan,nan,489 | 17 | 111 | 207 | 206 | 110 | 22 | 195 | ...,583 | 110 | 206 | 312 | 311 | 205 | 108 | 215 ...,584,nan,9,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,c-terminal domain of poly(a) binding protein |...,- | Nucleotide-binding alpha-beta plait domain...
4934,ENSG00000128342_ProteinCoding_1,inclusion,-0.239586841772764,True,LIF,ENST00000249075,chr22,30244755,30244933,179,...,6.0,65.0,23 | 43 | 3 | 33 | 15 | 1 | 34 | 24,202 | 14 | 32 | 191 | 22,202,False,9,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,- | leukemia inhibitory factor | LIF / OSM fam...,"Four-helical cytokine-like, core | - | Leukemi..."
6940,ENSG00000144355_ProteinCoding_1,inclusion,-0.2422743850073355,True,DLX1,ENST00000361725,chr2,172086654,172086853,200,...,104.0,170.0,123 | 129 | 95 | 100 | 128 | 125 | 161 | 126,189 | 185 | 118 | 112 | 190 | 184 | 186,255,False,9,CATH-Gene3D | CATH-FunFam | CDD | MobiDB-lite ...,Homeodomain-like | Distal-less homeobox 1 | - ...,- | Homeodomain | Homeodomain-like superfamily...
5902,ENSG00000136717_ProteinCoding_6,exon_diff_junction_siblings,-0.2212741177486916,True,BIN1,ENST00000393040 | ENST00000346226,chr2,127057473,127057601,129,...,nan,nan,10 | 400 | 157 | 222 | 31 | 410 | 249 | 342 | ...,245 | 482 | 191 | 242 | 241 | 481 | 323 | 377 ...,482 | 518,nan,9,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Arfaptin homology (AH) domain/BAR domain | SH3...,"AH/BAR domain superfamily | - | Amphiphysin 2,..."
3976,ENSG00000116337_ProteinCoding_2,exon_diff_junction_siblings,-0.4510693963749933,True,AMPD2,ENST00000528667 | ENST00000528454,chr1,109625303,109625433,131,...,nan,nan,270 | 371 | 400 | 99 | 301 | 1 | 11 | 55 | 163...,796 | 461 | 119 | 797 | 49 | 21 | 814 | 806 | ...,825 | 761,nan,9,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Metal-dependent hydrolases | - | AMP deaminase...,- | AMP deaminase | Metal-dependent hydrolase ...
4032,ENSG00000116983_ProteinCoding_1,inclusion,-0.2819875540122913,True,HPCAL4,ENST00000372844,chr1,3968393

In [29]:
for ct, df in df_list.items():
    if df['transcript_id'].isin(["ENST00000556803", "ENST00000556803"]).any():
        print(ct)